# Google Colab Vector Indexing Guide
## Build FAISS Index with GPU Acceleration

This notebook guides you through building a vector index for MS MARCO passages using Google Colab's **free GPU**.

**Timeline:**
- 10K documents: ~5 minutes
- 100K documents: ~30-60 minutes  
- 1M documents: ~5-10 hours

**⚠️ Important:** Colab sessions have 12-hour limits. Use checkpointing to resume if needed.

## STEP 1: Enable GPU in Google Colab

1. Click **Runtime** in the menu bar
2. Click **Change runtime type**
3. Select **GPU** from the "Hardware accelerator" dropdown
4. Click **Save**

This will restart your notebook with GPU support. You should see "GPU" in the top-right corner of Colab.

## STEP 2: Install Required Dependencies

In [ ]:
# Install required packages
!pip install -q sentence-transformers faiss-gpu numpy tqdm python-dotenv

## STEP 3: Upload Collection Data

You have two options:

**Option A: Upload from your computer (if < 100MB)**
1. Click folder icon on the left sidebar
2. Click upload file icon (↑)
3. Select your `collection.tsv`
4. It will appear in `/content/` directory

**Option B: Use Google Drive**
1. First, mount Google Drive:
2. Upload `collection.tsv` to your Google Drive
3. Then access it via `/content/gdrive/MyDrive/...`

We'll show Option A below.

In [ ]:
# Check if collection.tsv is uploaded
import os
from pathlib import Path

collection_path = Path('/content/collection.tsv')

if collection_path.exists():
    size_mb = collection_path.stat().st_size / (1024 * 1024)
    print(f"✓ Found collection.tsv ({size_mb:.1f} MB)")
else:
    print("⚠ collection.tsv not found at /content/collection.tsv")
    print("Please upload it using the file upload button on the left sidebar")
    print("Or use Google Drive option above")

## STEP 4: Clone Repository and Setup

In [ ]:
# Clone your repository
import os
os.chdir('/content')

# Replace with your actual repository URL
repo_url = "https://github.com/YOUR_USERNAME/hybrid-search-engine.git"

# Clone (skip if already cloned)
if not os.path.exists('/content/hybrid-search-engine'):
    !git clone {repo_url}
    print("✓ Repository cloned")
else:
    print("✓ Repository already exists")

os.chdir('/content/hybrid-search-engine')
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Setup directories and copy collection.tsv
import shutil
from pathlib import Path

# Create data directories
data_dir = Path('/content/hybrid-search-engine/data')
data_dir.mkdir(exist_ok=True)
msmarco_dir = data_dir / 'msmarco'
msmarco_dir.mkdir(exist_ok=True)

# Copy uploaded collection.tsv to the right place
src = Path('/content/collection.tsv')
dst = msmarco_dir / 'collection.tsv'

if src.exists() and not dst.exists():
    shutil.copy(src, dst)
    print(f"✓ Collection copied to {dst}")
elif dst.exists():
    print(f"✓ Collection already at {dst}")
else:
    print("⚠ collection.tsv not found. Please upload it first.")

# Create indexes directory
indexes_dir = data_dir / 'indexes'
indexes_dir.mkdir(exist_ok=True)
print(f"✓ Created indexes directory: {indexes_dir}")

## STEP 5: Run Vector Indexing

Start with **Stage 1: Small Test (10K documents)** to validate everything works before committing to larger runs.

After verification, progress to larger stages.

### Stage 1: Test with Small Dataset (10K documents)

This validates the entire pipeline. Run this first.

In [ ]:
!python scripts/index_vectors.py \
    --collection data/msmarco/collection.tsv \
    --max-docs 10000 \
    --reset

### Stage 2: Medium Dataset (100K documents)

After Stage 1 succeeds, run this to build a larger index. Resume is automatic.

In [ ]:
# Uncomment to run Stage 2 (only after Stage 1 completes successfully)
# !python scripts/index_vectors.py \
#     --collection data/msmarco/collection.tsv \
#     --max-docs 100000

### Stage 3: Large Dataset (1M documents)

For full production indexing. Can take several hours.

In [ ]:
# Uncomment to run Stage 3 (only after Stage 2 completes successfully)
# !python scripts/index_vectors.py \
#     --collection data/msmarco/collection.tsv \
#     --max-docs 1000000

## STEP 6: Monitor Indexing Progress

Check progress without re-running the indexing command. Use this anytime to see current status.

In [ ]:
!python scripts/index_vectors.py --status

## STEP 7: Download Built Index to Your Computer

After indexing completes, download these two files:
1. `vector.faiss` - The FAISS index
2. `vector_checkpoint.json` - Metadata for resuming

In [ ]:
from google.colab import files
from pathlib import Path

index_dir = Path('/content/hybrid-search-engine/data/indexes')

# Download FAISS index
faiss_path = index_dir / 'vector.faiss'
checkpoint_path = index_dir / 'vector_checkpoint.json'

print("Downloading files...")
if faiss_path.exists():
    files.download(str(faiss_path))
    print(f"✓ Downloaded vector.faiss ({faiss_path.stat().st_size / (1024*1024):.1f} MB)")

if checkpoint_path.exists():
    files.download(str(checkpoint_path))
    print("✓ Downloaded vector_checkpoint.json")

print("\nFiles are now in your Downloads folder!")

## Final Steps: Use Index Locally

After downloading `vector.faiss`, copy it to your local project:

```
c:\Users\golde\Desktop\Projects\Hybrid_search_engine\hybrid-search-engine\data\indexes\vector.faiss
```

Then test it locally:
```bash
python scripts/test_vector.py
```

## Troubleshooting

**Q: Session disconnected during indexing?**
- Your checkpoint is saved! Just re-run the same command and it will resume automatically.

**Q: Out of memory errors?**
- Reduce batch sizes: `--batch-size 256 --encode-batch-size 32`
- Reduce `--max-docs` for testing

**Q: GPU not working?**
- Check Runtime > Change runtime type shows "GPU"
- Run this to verify: `!nvidia-smi`

**Q: Too slow even with GPU?**
- This is normal for large datasets. 100K documents ≈ 30-60 min on T4 GPU
- For faster speeds, use batching and stick with GPU

**Q: How to resume if Colab crashes?**
- The checkpoint is saved in `data/indexes/vector_checkpoint.json`
- Just re-run: `!python scripts/index_vectors.py --collection data/msmarco/collection.tsv --max-docs 100000`
- It will automatically skip already indexed documents